In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from scipy.stats import spearmanr

print('All imports successful')

All imports successful


In [2]:
from pathlib import Path

BASE = Path('.')

TRAIN_FILE      = str(BASE / 'train.csv')
VAL_FILE        = str(BASE / 'val.csv')
TEST_FILE       = str(BASE / 'test.csv')
SAMPLE_SUB_FILE = str(BASE / 'SampleSubmission.csv')
SUBMISSION_FILE = str(BASE / 'submission.csv')

ID_COL      = 'ID'
TARGET_COLS = [
    'no_water_access', 'no_sanitation_access', 'no_refuse_access',
    'no_energy_access', 'no_education_access',
]

RANDOM_STATE = 42
print('Config loaded')

Config loaded


In [3]:
train      = pd.read_csv(TRAIN_FILE)
val        = pd.read_csv(VAL_FILE)
test       = pd.read_csv(TEST_FILE)
sample_sub = pd.read_csv(SAMPLE_SUB_FILE)

print(f'Train : {train.shape}')
print(f'Val   : {val.shape}')
print(f'Test  : {test.shape}')

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [4]:
def engineer_features(df):
    df = df.copy()

    # ── Ordinal bands → integers ──────────────────────────────────────────────
    df['asset_index']      = df['asset_index'].map(
        {'0-2': 0, '3-4': 1, '5-6': 2, '7-8': 3, '9+': 4})
    df['debt_count']       = df['debt_count'].map(
        {'0': 0, '1': 1, '2': 2, '3+': 3})
    df['income_src_count'] = df['income_src_count'].map(
        {'0': 0, '1': 1, '2+': 2})
    df['hazard_count']     = df['hazard_count'].map(
        {'0': 0, '1': 1, '2': 2, '3+': 3})
    df['amenity_access']   = df['amenity_access'].map(
        {'0-2': 0, '3-4': 1, '5-6': 2, '7+': 3})
    df['chronic_count']    = df['chronic_count'].map(
        {'0': 0, '1': 1, '2+': 2})
    df['hh_size_band']     = df['hh_size_band'].map(
        {'1': 1, '2': 2, '3-4': 3, '5-6': 5, '7+': 7})
    df['rooms_band']       = df['rooms_band'].map(
        {'1-2': 1, '3-4': 3, '5+': 5})
    df['children_band']    = df['children_band'].map(
        {'0': 0, '1': 1, '2': 2, '3+': 3})
    df['age_band']         = df['age_band'].map(
        {'<25': 0, '25-34': 1, '35-44': 2, '45-54': 3,
         '55-64': 4, '65+': 5, '__SUPPRESSED__': np.nan})
    df['tenure']           = df['tenure'].map(
        {'Owned': 0, 'Renting': 1,
         'Allowed to stay rent free by owner': 2,
         'Squatting or living rent-free in an informal dwelling youve built, or in a vacant building or on vacant land': 3})
    df['transport_mode']   = df['transport_mode'].map(
        {'Car as driver': 0, 'Car as passenger': 1,
         'Minibus Taxi': 2, 'Walk': 3})
    df['health_status']    = df['health_status'].map(
        {'Excellent': 0, 'Good': 1, 'Poor': 2})
    df['marital']          = df['marital'].map(
        {'Married/common law marriage': 0,
         'In a relationship but not married': 1,
         'Single': 2, 'Divorced': 3, 'Widowed': 4})

    # ── Binary columns → 0/1 ─────────────────────────────────────────────────
    for col in ['internet', 'employed', 'social_grant', 'medical_aid', 'crime_victim']:
        df[col] = (df[col] == 'Yes').astype(int)

    df['food_insecure']    = (df['food_insecurity'] == 'insecure').astype(int)
    df['is_informal']      = (df['dwelling_type'] == 'Informal').astype(int)
    df['is_female']        = (df['sex'] == 'Female').astype(int)
    df['is_head']          = (df['head_household'] == 'Respondent').astype(int)
    df['is_black_african'] = (df['pop_group'] == 'Black African').astype(int)

    # Drop original string columns now replaced
    df = df.drop(columns=['dwelling_type', 'food_insecurity', 'sex',
                           'head_household', 'pop_group'])

    # ── Composite scores ──────────────────────────────────────────────────────
    hh = df['hh_size_band'] + 1

    df['informal_score']    = (
        df['is_informal'] +
        (df['tenure'] >= 2).astype(int)
    )
    df['ses_vulnerability'] = (
        (df['employed'] == 0).astype(int) +
        (df['medical_aid'] == 0).astype(int) +
        (df['internet'] == 0).astype(int) +
        df['social_grant'] +
        (df['asset_index'] <= 1).astype(int) +
        df['food_insecure'] +
        (df['income_src_count'] == 0).astype(int)
    )
    df['crowding_proxy']   = df['hh_size_band'] / (df['rooms_band'] + 1)
    df['dependents_total'] = df['children_band'] + df['elderly_band']
    df['isolation_score']  = (
        (df['internet'] == 0).astype(int) +
        (df['transport_mode'] >= 2).astype(int) +
        (df['amenity_access'] <= 1).astype(int) +
        (df['medical_aid'] == 0).astype(int)
    )
    df['health_burden']    = (
        df['chronic_count'] +
        df['hazard_count'] +
        df['health_status']
    )

    # ── Per capita features ───────────────────────────────────────────────────
    df['assets_per_person'] = df['asset_index'] / hh
    df['rooms_per_person']  = df['rooms_band'] / hh
    df['debt_count_per_person'] = df['debt_count'] / hh

    # ── Interaction terms ─────────────────────────────────────────────────────
    df['informal_x_ses']   = df['informal_score'] * df['ses_vulnerability']
    df['asset_x_formal']   = df['asset_index'] * (1 - df['is_informal'])
    df['asset_x_informal'] = df['asset_index'] * df['is_informal']
    df['walk_x_informal']  = (df['transport_mode'] == 3).astype(int) * df['is_informal']
    df['crowded_informal'] = df['crowding_proxy'] * df['is_informal']
    df['ses_x_isolation']  = df['ses_vulnerability'] * df['isolation_score']
    df['female_informal']  = df['is_female'] * df['is_informal']

    # ── Team features ─────────────────────────────────────────────────────────
    df['triple_zero'] = (
        (df['income_src_count'] == 0) &
        (df['asset_index'] == 0) &
        (df['amenity_access'] == 0)
    ).astype(int)

    df['no_safety_net'] = (
        (df['is_informal'] == 1) &
        (df['employed'] == 0) &
        (df['social_grant'] == 0)
    ).astype(int)

    df['is_transport_vulnerable'] = (df['transport_mode'] >= 2).astype(int)

    df['total_vulnerable_members'] = (
        df['children_band'] +
        df['chronic_count'] +
        df['elderly_band']
    )

    df['hh_size_mid'] = df['hh_size_band'].map(
        {1: 1, 2: 2, 3: 3.5, 5: 5.5, 7: 9.0})
    df['rooms_mid']   = df['rooms_band'].map(
        {1: 1.5, 3: 3.5, 5: 6.0})
    df['crowding_mid'] = df['hh_size_mid'] / (df['rooms_mid'] + 1)

    return df

print('Feature engineering function defined')

Feature engineering function defined


In [5]:
# Reload raw data to ensure clean state
train = pd.read_csv(TRAIN_FILE)
val   = pd.read_csv(VAL_FILE)
test  = pd.read_csv(TEST_FILE)

features_to_drop = [
    'crime_victim',
    'is_white',
    'debt_per_person',
    'dependency_burden',
    'income_sources_per_person',
]

X_train = engineer_features(train.drop(columns=TARGET_COLS + [ID_COL]))
X_val   = engineer_features(val.drop(columns=TARGET_COLS + [ID_COL]))
X_test  = engineer_features(test.drop(columns=[ID_COL]))

y_train = train[TARGET_COLS].values.astype(int)
y_val   = val[TARGET_COLS].values.astype(int)

X_train = X_train.drop(columns=features_to_drop, errors='ignore')
X_val   = X_val.drop(columns=features_to_drop, errors='ignore')
X_test  = X_test.drop(columns=features_to_drop, errors='ignore')

print(f'X_train : {X_train.shape}')
print(f'X_val   : {X_val.shape}')
print(f'X_test  : {X_test.shape}')
print(f'\nNull counts:')
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])
print(f'\nFeatures: {list(X_train.columns)}')

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [6]:
rf_model = MultiOutputClassifier(
    RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
)

rf_model.fit(X_train, y_train)

rf_preds     = rf_model.predict(X_val)
rf_macro     = f1_score(y_val, rf_preds, average='macro', zero_division=0)
rf_per_label = f1_score(y_val, rf_preds, average=None, zero_division=0)

print(f'RF Macro-F1 @ 0.5 : {rf_macro:.4f}')
print(f'Baseline was      : 0.4169')
print()
print('Per-label F1:')
for col, score in zip(TARGET_COLS, rf_per_label):
    print(f'  {col:<25} {score:.4f}')

NameError: name 'X_train' is not defined

In [7]:
rf_probs    = np.column_stack([p[:, 1] for p in rf_model.predict_proba(X_val)])
best_thr_rf = {}
tuned_f1_rf = {}

print(f'  {"label":<25} {"thr":>5}  {"F1@0.5":>7}  {"F1@thr":>7}  {"delta":>7}')

for i, target in enumerate(TARGET_COLS):
    base_f1 = f1_score(y_val[:, i],
                       (rf_probs[:, i] >= 0.5).astype(int),
                       zero_division=0)
    best_f1, best_thr = 0, 0.5
    for thr in np.arange(0.05, 0.95, 0.01):
        preds = (rf_probs[:, i] >= thr).astype(int)
        f1    = f1_score(y_val[:, i], preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, thr
    best_thr_rf[target] = round(best_thr, 2)
    tuned_f1_rf[target] = best_f1
    print(f'  {target:<25} {best_thr:>5.2f}  {base_f1:>7.4f}  {best_f1:>7.4f}  {best_f1-base_f1:>+7.4f}')

print(f'\nRF Tuned Macro-F1 : {np.mean(list(tuned_f1_rf.values())):.4f}')
print(f'Baseline was      : 0.4169')

NameError: name 'X_val' is not defined

In [8]:
rf_test_probs = np.column_stack([p[:, 1] for p in rf_model.predict_proba(X_test)])

submission = sample_sub[[ID_COL]].copy()
for i, target in enumerate(TARGET_COLS):
    submission[target] = (rf_test_probs[:, i] >= best_thr_rf[target]).astype(int)

assert list(submission.columns) == [ID_COL] + TARGET_COLS
assert len(submission) == len(sample_sub)
assert submission.isna().sum().sum() == 0
for col in TARGET_COLS:
    assert submission[col].isin([0, 1]).all()

print('All sanity checks passed.')
print(f'Submission shape: {submission.shape}')
print('\nPredicted positives per label:')
print((submission[TARGET_COLS].mean() * 100).apply(lambda x: f'{x:.1f}%'))

submission.to_csv(SUBMISSION_FILE, index=False)
print(f'\nSubmission saved to: {SUBMISSION_FILE}')

NameError: name 'X_test' is not defined

In [9]:
import os
print("Current directory:", os.getcwd())
print("\nFiles here:")
print(os.listdir('.'))

Current directory: C:\Users\Latitude 3420

Files here:
['.anaconda', '.android', '.cache', '.conda', '.condarc', '.config', '.continuum', '.copilot', '.cursor', '.eclipse', '.emulator_console_auth_token', '.gitconfig', '.gradle', '.ipynb_checkpoints', '.ipython', '.jupyter', '.kaggle', '.lesshst', '.matplotlib', '.node_repl_history', '.p2', '.prefect', '.viminfo', '.virtualenvs', '.vscode', '.vscode-shared', 'anaconda3', 'AndroidStudioProjects', 'AppData', 'Application Data', 'A_star_plots.png', 'cls', 'Contacts', 'Cookies', 'DA.ipynb', 'Desktop', 'Documents', 'Downloads', 'eclipse', 'eclipse-workspace', 'Favorites', 'Figure_1.png', 'GBFS_plots.png', 'IdeaProjects', 'IntelGraphicsProfiles', 'Links', 'Local Settings', 'manim_renders', 'media', 'mfput.log', 'miktex-console.lock', 'mushroom analysis', 'Music', 'My Documents', 'NetHood', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{fa401041-fefe-11ef-802d-b1a4edfdc667}.TM.blf', 'NTUSER.DAT{fa401041-fefe-11ef-802d-b1a4edf